In [17]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("output/test_count_aggregate.csv")
df = pd.read_csv(CSV_PATH)

print(df.shape)
df.head()


(801, 11)


,Model,Context,Instance,TestFileCount,TotalTestCount,CompiledTestFileCount,CompiledTestCount,ExecutedTestFileCount,ExecutedTestCount,DetectedTestFileCount,DetectedBCTestCount
0,GPT4o,Class,BBC01,1,1,NOT_COMPILED,NOT_COMPILED,NOT_VALID,NOT_VALID,NOT_VALID,NOT_VALID
1,GPT4o,Class,BBC02,2,2,NOT_COMPILED,NOT_COMPILED,NOT_VALID,NOT_VALID,NOT_VALID,NOT_VALID
2,GPT4o,Class,BBC03,2,2,1,1,1,1,1,1
3,GPT4o,Class,BBC04,51,91,1,1,1,1,0,0
4,GPT4o,Class,BBC05,4,6,NOT_COMPILED,NOT_COMPILED,NOT_VALID,NOT_VALID,NOT_VALID,NOT_VALID


In [18]:
# confirm which columns have string values
print(df.dtypes)
print()
for col in ["CompiledTestFileCount", "CompiledTestCount", "ExecutedTestFileCount",
            "ExecutedTestCount", "DetectedTestFileCount", "DetectedBCTestCount"]:
    print(col, "->", df[col].unique()[:5])


Model                    object
Context                  object
Instance                 object
TestFileCount             int64
TotalTestCount            int64
CompiledTestFileCount    object
CompiledTestCount        object
ExecutedTestFileCount    object
ExecutedTestCount        object
DetectedTestFileCount    object
DetectedBCTestCount      object
dtype: object

CompiledTestFileCount -> ['NOT_COMPILED' '1' '49' '2' '247']
CompiledTestCount -> ['NOT_COMPILED' '1' '119' '2' '492']
ExecutedTestFileCount -> ['NOT_VALID' '1' '16' '0' '167']
ExecutedTestCount -> ['NOT_VALID' '1' '36' '0' '270']
DetectedTestFileCount -> ['NOT_VALID' '1' '0' '16' '2']
DetectedBCTestCount -> ['NOT_VALID' '1' '0' '36' '2']


Adding 0 as cannot drop the rows, it will miscalculate stuff.

In [19]:
SENTINELS = {"NOT_COMPILED": 0, "NOT_VALID": 0}
NUMERIC_COLS = ["TestFileCount", "TotalTestCount", "CompiledTestFileCount", "CompiledTestCount",
                "ExecutedTestFileCount", "ExecutedTestCount", "DetectedTestFileCount", "DetectedBCTestCount"]

df_clean = df.copy()
for col in NUMERIC_COLS:
    df_clean[col] = df_clean[col].replace(SENTINELS).astype(int)

df_clean.dtypes


Model                    object
Context                  object
Instance                 object
TestFileCount             int64
TotalTestCount            int64
CompiledTestFileCount     int64
CompiledTestCount         int64
ExecutedTestFileCount     int64
ExecutedTestCount         int64
DetectedTestFileCount     int64
DetectedBCTestCount       int64
dtype: object

In [20]:
NUMERIC_COLS = ["TestFileCount", "TotalTestCount", "CompiledTestFileCount", "CompiledTestCount",
                "ExecutedTestFileCount", "ExecutedTestCount", "DetectedTestFileCount", "DetectedBCTestCount"]

modelcontext_df = (
    df_clean
    .groupby(["Model", "Context"], as_index=False)[NUMERIC_COLS]
    .sum()
)

modelcontext_df


,Model,Context,TestFileCount,TotalTestCount,CompiledTestFileCount,CompiledTestCount,ExecutedTestFileCount,ExecutedTestCount,DetectedTestFileCount,DetectedBCTestCount
0,GPT4o,Class,5790,9841,1868,3178,958,1511,320,437
1,GPT4o,Method,5790,10252,1684,2845,759,1091,292,384
2,GPT4o,Minimal,5790,6904,1014,1146,412,445,168,172
3,GPTOSS,Class,5790,12382,214,396,149,309,67,107
4,GPTOSS,Method,5790,11287,199,325,107,191,77,125
5,GPTOSS,Minimal,5790,8419,117,157,23,30,7,9
6,Qwen3-coder,Class,5790,14221,1939,4043,1276,2483,707,887
7,Qwen3-coder,Method,5790,12963,1802,3142,949,1529,614,782
8,Qwen3-coder,Minimal,5790,10283,1721,2291,863,1149,649,692


In [5]:
df_clean.head()
print(df_clean.shape)  # should still be (802, 11)


(801, 11)


In [21]:
modelcontext_df = (
    df_clean
    .groupby(["Model", "Context"])
    .agg(
        InstanceCount=("Instance", "nunique"),
        **{col: (col, "sum") for col in NUMERIC_COLS}
    )
    .reset_index()
)

modelcontext_df


,Model,Context,InstanceCount,TestFileCount,TotalTestCount,CompiledTestFileCount,CompiledTestCount,ExecutedTestFileCount,ExecutedTestCount,DetectedTestFileCount,DetectedBCTestCount
0,GPT4o,Class,89,5790,9841,1868,3178,958,1511,320,437
1,GPT4o,Method,89,5790,10252,1684,2845,759,1091,292,384
2,GPT4o,Minimal,89,5790,6904,1014,1146,412,445,168,172
3,GPTOSS,Class,89,5790,12382,214,396,149,309,67,107
4,GPTOSS,Method,89,5790,11287,199,325,107,191,77,125
5,GPTOSS,Minimal,89,5790,8419,117,157,23,30,7,9
6,Qwen3-coder,Class,89,5790,14221,1939,4043,1276,2483,707,887
7,Qwen3-coder,Method,89,5790,12963,1802,3142,949,1529,614,782
8,Qwen3-coder,Minimal,89,5790,10283,1721,2291,863,1149,649,692


In [22]:
df_clean["ReachedGenerated"] = df_clean["TestFileCount"] > 0
df_clean["ReachedCompiled"]  = df_clean["CompiledTestFileCount"] > 0
df_clean["ReachedExecuted"]  = df_clean["ExecutedTestFileCount"] > 0
df_clean["ReachedDetectedBC"] = df_clean["DetectedBCTestCount"] > 0

df_clean[["Model", "Context", "Instance", "ReachedGenerated", "ReachedCompiled", "ReachedExecuted", "ReachedDetectedBC"]].head()


,Model,Context,Instance,ReachedGenerated,ReachedCompiled,ReachedExecuted,ReachedDetectedBC
0,GPT4o,Class,BBC01,True,False,False,False
1,GPT4o,Class,BBC02,True,False,False,False
2,GPT4o,Class,BBC03,True,True,True,True
3,GPT4o,Class,BBC04,True,True,True,False
4,GPT4o,Class,BBC05,True,False,False,False


In [23]:
STAGE_FLAGS = ["ReachedGenerated", "ReachedCompiled", "ReachedExecuted", "ReachedDetectedBC"]

modelcontext_df = (
    df_clean
    .groupby(["Model", "Context"])
    .agg(
        InstanceCount=("Instance", "nunique"),
        InstancesGenerated=("ReachedGenerated", "sum"),
        InstancesCompiled=("ReachedCompiled", "sum"),
        InstancesExecuted=("ReachedExecuted", "sum"),
        InstancesDetectedBC=("ReachedDetectedBC", "sum"),
        **{col: (col, "sum") for col in NUMERIC_COLS}
    )
    .reset_index()
)

modelcontext_df


,Model,Context,InstanceCount,InstancesGenerated,InstancesCompiled,InstancesExecuted,InstancesDetectedBC,TestFileCount,TotalTestCount,CompiledTestFileCount,CompiledTestCount,ExecutedTestFileCount,ExecutedTestCount,DetectedTestFileCount,DetectedBCTestCount
0,GPT4o,Class,89,89,39,29,23,5790,9841,1868,3178,958,1511,320,437
1,GPT4o,Method,89,89,39,32,11,5790,10252,1684,2845,759,1091,292,384
2,GPT4o,Minimal,89,89,42,36,15,5790,6904,1014,1146,412,445,168,172
3,GPTOSS,Class,89,89,24,20,9,5790,12382,214,396,149,309,67,107
4,GPTOSS,Method,89,89,20,17,8,5790,11287,199,325,107,191,77,125
5,GPTOSS,Minimal,89,89,24,8,3,5790,8419,117,157,23,30,7,9
6,Qwen3-coder,Class,89,89,37,32,16,5790,14221,1939,4043,1276,2483,707,887
7,Qwen3-coder,Method,89,89,37,32,18,5790,12963,1802,3142,949,1529,614,782
8,Qwen3-coder,Minimal,89,89,36,29,13,5790,10283,1721,2291,863,1149,649,692
